# Cross-table relationships & summary

Run this last, once `DESCRIBE` output from notebooks 01-05 shows the real column names — the queries below use placeholder column names that need fixing first.

## Run 1 — initial exploration ⚠️ *(run in Databricks — both queries were stale placeholders; see Run 2 below for working versions)*

## How many vendors per RFQ?
Real column names confirmed in `02_rfqvendor.ipynb`: `RFQNUM`, `VENDOR`.

In [ ]:
%sql
SELECT RFQNUM, COUNT(DISTINCT VENDOR) AS vendor_count
FROM ingestion_framework_test.bid_data_exploration.rfqvendor
GROUP BY RFQNUM
ORDER BY vendor_count DESC
LIMIT 30

## D-111808: why 22 invited vendors but only 8 priced lines in `quotationline`?

`rfqvendor` shows 22 vendors invited to D-111808; `quotationline` shows only 8 rows (see `FINDINGS.md`'s headline finding). This joins the two on the real composite key `(RFQNUM, VENDOR)` — confirmed in `03_quotationline.ipynb` — to see exactly which invited vendors never produced a priced line, and whether their `BIDSTATUS` explains why (e.g. `REGRETTED`/`NOT SIGNED` vendors should have zero lines; `SUBMITTED` vendors with zero lines would be a real gap worth flagging).

In [ ]:
%sql
SELECT rv.RFQNUM, rv.VENDOR, rv.BIDSTATUS, COUNT(ql.QUOTATIONLINEID) AS line_count
FROM ingestion_framework_test.bid_data_exploration.rfqvendor rv
LEFT JOIN ingestion_framework_test.bid_data_exploration.quotationline ql
  ON rv.RFQNUM = ql.RFQNUM AND rv.VENDOR = ql.VENDOR
WHERE rv.RFQNUM = 'D-111808'
GROUP BY rv.RFQNUM, rv.VENDOR, rv.BIDSTATUS
ORDER BY line_count DESC

## Does `DETAILBOQAVAILABLE` actually correlate with real itemized BOQ data?

`rfq.DETAILBOQAVAILABLE` is populated on only ~1.8% of RFQs (Run 2, `01_rfq.ipynb`) — `Y` on 412, `N` on 141. Before treating `Y` as a meaningful "this tender has real itemized pricing" signal, check whether it actually lines up with `quotationline` having populated, multi-line `BOQITEMNUM` data — or whether the two are unrelated.

### For RFQs flagged `DETAILBOQAVAILABLE = 'Y'`, does `quotationline` actually show itemized (multi-`BOQITEMNUM`) data?

In [ ]:
%sql
SELECT r.RFQNUM, r.DESCRIPTION,
       COUNT(DISTINCT ql.BOQITEMNUM) AS distinct_boqitems,
       COUNT(ql.QUOTATIONLINEID) AS total_lines
FROM ingestion_framework_test.bid_data_exploration.rfq r
LEFT JOIN ingestion_framework_test.bid_data_exploration.quotationline ql ON r.RFQNUM = ql.RFQNUM
WHERE r.DETAILBOQAVAILABLE = 'Y'
GROUP BY r.RFQNUM, r.DESCRIPTION
ORDER BY distinct_boqitems DESC
LIMIT 30

### Conversely — do `null`-flagged RFQs ever have real itemized `BOQITEMNUM` data anyway? (tests whether `null` really means "no BOQ", or just "not assessed")

In [ ]:
%sql
SELECT r.RFQNUM, r.DETAILBOQAVAILABLE, COUNT(DISTINCT ql.BOQITEMNUM) AS distinct_boqitems
FROM ingestion_framework_test.bid_data_exploration.rfq r
JOIN ingestion_framework_test.bid_data_exploration.quotationline ql ON r.RFQNUM = ql.RFQNUM
WHERE r.DETAILBOQAVAILABLE IS NULL AND ql.BOQITEMNUM IS NOT NULL
GROUP BY r.RFQNUM, r.DETAILBOQAVAILABLE
ORDER BY distinct_boqitems DESC
LIMIT 30

## Summary — fill in once all notebooks have been run

| Question | Answer |
|---|---|
| Which RFQ maps to a real tender like D-111808? | |
| Does one RFQ span multiple lots, or is a lot its own RFQ/line grouping? | |
| How are negotiation rounds represented? | |
| Does `quotationline` cleanly split CIF vs. Erection like the sample BOQs, or is pricing structured differently? | |
| Can bid documents (Excel/PDF) actually be retrieved via `docinfo`/`doclinks`, or is this metadata-only? | |
| Biggest blocker to reusing the existing extraction/comparison pipeline against this data? | |